# 02 — YOLOv8 Model Training

Train YOLOv8 on the labeled Harumanis dataset.

**Classes:** early (1), mid (2), late (3), preharvest (4)

**Steps:** setup → load pretrained model → train → evaluate → export weights

## 1. Setup

In [ ]:
!pip install ultralytics

import torch
from ultralytics import YOLO

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — use Google Colab GPU runtime for faster training.')

## 2. Configure paths and hyperparameters

In [ ]:
import os

DATA_YAML   = '../datasets/labeled/data.yaml'
WEIGHTS_DIR = '../weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

EPOCHS     = 100
IMG_SIZE   = 640
BATCH_SIZE = 16    # reduce to 8 if out of GPU memory
MODEL_BASE = 'yolov8n.pt'  # nano: fast, good for small datasets
               # options: yolov8s.pt (small), yolov8m.pt (medium)

print(f'Data : {DATA_YAML}')
print(f'Model: {MODEL_BASE}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}')

## 3. Train

In [ ]:
model = YOLO(MODEL_BASE)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name='mango_yolov8',
    project=WEIGHTS_DIR,
    patience=20,
    augment=True,
    degrees=15,
    flipud=0.3,
    fliplr=0.5,
    hsv_h=0.02,
    hsv_s=0.5,
    hsv_v=0.4,
)
print('Training complete.')
print(f'Best weights: {WEIGHTS_DIR}/mango_yolov8/weights/best.pt')

## 4. Evaluate on validation set

In [ ]:
best = YOLO(f'{WEIGHTS_DIR}/mango_yolov8/weights/best.pt')
metrics = best.val(data=DATA_YAML)

print(f'mAP@0.5      : {metrics.box.map50:.3f}')
print(f'mAP@0.5-0.95 : {metrics.box.map:.3f}')
print()
print('Per-class AP@0.5:')
for name, ap in zip(metrics.names.values(), metrics.box.ap50):
    print(f'  {name:15s}: {ap:.3f}  {chr(9608)*int(ap*20)}')

## 5. Visualise predictions

In [ ]:
import cv2, glob, matplotlib.pyplot as plt

val_imgs = glob.glob('../datasets/labeled/valid/images/*.jpg')[:4]
fig, axes = plt.subplots(1, len(val_imgs), figsize=(20, 5))
for ax, p in zip(axes, val_imgs):
    res = best.predict(p, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.suptitle('Predictions on validation images')
plt.tight_layout()
plt.show()

## 6. Copy weights for API deployment

In [ ]:
import shutil

src = f'{WEIGHTS_DIR}/mango_yolov8/weights/best.pt'
dst = f'{WEIGHTS_DIR}/mango_yolov8.pt'
shutil.copy(src, dst)

size_mb = os.path.getsize(dst) / 1024 / 1024
print(f'Saved: {dst}  ({size_mb:.1f} MB)')
print('Next: copy to mlharum-api/weights/mango_yolov8.pt')

## 7. Training log

Record results after each run.

| Run | Base model | Epochs | mAP@0.5 | Notes |
|-----|-----------|--------|---------|-------|
| 1   | yolov8n   | 100    |         |       |

**If mAP@0.5 < 0.5:** check annotation quality, class balance, try yolov8s.pt, collect more data for weak classes.